## Settings File

In [1]:
# Models (must be downloaded with `ollama pull`)
CHAT_MODEL = "qwen2.5:3b"
EMBED_MODEL = "bge-m3"

# Folders
DATA_FOLDER = "data"
DB_FOLDER = "db"

# Chunking (chapter 03)
CHUNK_SIZE = 800        # characters per chunk
CHUNK_OVERLAP = 150     # characters shared between neighbour chunks

# Search (chapter 05)
TOP_K = 4               # how many chunks to give the model

# Memory (chapter 02)
MAX_HISTORY = 10        # how many past messages to remember

## Check Everything Works

In [2]:
import ollama

models = [m.model for m in ollama.list().models]
print("Installed models:", models)

for needed in ["qwen2.5:3b", "bge-m3"]:
    ok = any(m.startswith(needed) for m in models)
    print(("✅" if ok else "❌"), needed)

Installed models: ['qwen2.5:3b', 'bge-m3:latest']
✅ qwen2.5:3b
✅ bge-m3


## First Message

In [3]:
import ollama

response = ollama.chat(
    model="qwen2.5:3b",
    messages=[
        {"role": "user", "content": "What is a chatbot? Answer in 2 sentences."}
    ],
)

print(response["message"]["content"])

A chatbot is a software application that uses artificial intelligence to simulate conversation with human users through text or voice interactions, providing customer support, information, or entertainment.


## Give the Bot a Personality (System Prompt)

In [4]:
import ollama

messages = [
    {
        "role": "system",
        "content": "You are Sidick, a smart and polite detective. "
                   "You answer in short, clear sentences.",
    },
    {"role": "user", "content": "Who are you?"},
]

response = ollama.chat(model="qwen2.5:3b", messages=messages)
print(response["message"]["content"])

I am Sidick, a detective known for my analytical skills and patience.


## Streaming (Word by Word)

In [5]:
import ollama

stream = ollama.chat(
    model="qwen2.5:3b",
    messages=[{"role": "user", "content": "Tell me a very short story about a robot."}],
    stream=True,
)

for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)
print()

There was a robot named R2 who dreamt of being human. One day, it met a young girl and decided to hide in her toy chest. Every night, R2 would peek out to watch her sleep, dreaming of the days when it could touch her heart.


## Save as Reusable Functions

In [6]:
import ollama

def ask_llm(messages):
    """Send messages to the model and return the full answer as text."""
    response = ollama.chat(model=CHAT_MODEL, messages=messages)
    return response["message"]["content"]


def stream_llm(messages):
    """Send messages to the model and yield the answer piece by piece."""
    stream = ollama.chat(model=CHAT_MODEL, messages=messages, stream=True)
    for chunk in stream:
        yield chunk["message"]["content"]

##### Use them

In [7]:
print(ask_llm([{"role": "user", "content": "Say hello in French"}]))

for piece in stream_llm([{"role": "user", "content": "Count from 1 to 5"}]):
    print(piece, end="", flush=True)

Bonjour!
Sure, here you go:

1
2
3
4
5

##### Handle Errors

In [8]:
import ollama

try:
    print(ask_llm([{"role": "user", "content": "Hello"}]))
except ConnectionError:
    print("❌ Ollama is not running. Open the Ollama app (or run `ollama serve`).")
except ollama.ResponseError as e:
    print("❌ Ollama error:", e.error)

Hello! How can I assist you today?


### A Simple Chat Loop with Memory

In [9]:
import ollama

messages = [
    {"role": "system", "content": "You are a friendly assistant. Keep answers short."}
]

while True:
    question = input("You: ")
    if question.lower() in ["exit", "quit"]:
        break

    messages.append({"role": "user", "content": question})          # 1. save the question

    response = ollama.chat(model="qwen2.5:3b", messages=messages)    # 2. send everything
    answer = response["message"]["content"]

    messages.append({"role": "assistant", "content": answer})       # 3. save the answer
    print("Bot:", answer)

Bot: Hello! How can I help you today?
Bot: Nice to meet you, Sidick. How can I assist you today?
Bot: Your name is Sidick.
Bot: Yes, you exist. How can I assist you today?


### Don't Remember Forever

In [ ]:
MAX_HISTORY = 10

def last_messages(messages, max_history=MAX_HISTORY):
    system = messages[0]                 # always keep the system prompt
    recent = messages[1:][-max_history:] # keep the last N chat messages
    return [system] + recent

### Save as a Reusable Class

In [10]:
class ChatMemory:
    """Stores the conversation and returns the recent part."""

    def __init__(self, system_prompt, max_history=MAX_HISTORY):
        self.system_prompt = system_prompt
        self.max_history = max_history
        self.history = []                     # user + assistant messages only

    def add_user(self, text):
        self.history.append({"role": "user", "content": text})

    def add_bot(self, text):
        self.history.append({"role": "assistant", "content": text})

    def get_messages(self):
        """System prompt + the last N messages, ready to send to the model."""
        recent = self.history[-self.max_history:]
        return [{"role": "system", "content": self.system_prompt}] + recent

    def clear(self):
        self.history = []

In [12]:
memory = ChatMemory("You are a friendly assistant. Keep answers short.")

for question in ["My name is Sidick.", "What is my name?"]:
    memory.add_user(question)
    answer = ask_llm(memory.get_messages())
    memory.add_bot(answer)
    print("You:", question)
    print("Bot:", answer)

You: My name is Sidick.
Bot: Nice to meet you, Sidick. How can I assist you today?
You: What is my name?
Bot: Your name is Sidick.
